# Raw Data Split


## 2. Install Required Packages

In [ ]:
# =========================================================
# 2. Install Required Packages
# =========================================================

!pip install dice-ml

## 3. Import Required Libraries

In [6]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

## 4. Mount Google Drive

In [7]:
# =========================================================
# 4. Mount Google Drive
# =========================================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 5. Define project paths

In [8]:
# =========================================================
# 5. Define Project Paths (Colab + Google Drive)
# =========================================================

# Root (Google Drive)
ROOT_PATH = Path("/content/drive/MyDrive/Master_Research")

# =====================================================
# Data Paths
# =====================================================

DATA_PATH = ROOT_PATH / "04_Data" / "01_Raw_Data" / "german_credit" / "german_credit_data.csv"
DATASET_DIR = ROOT_PATH / "07_Results" / "Datasets" / "german_credit"

# ---------------------------------------------------------
# Debug Prints
# ---------------------------------------------------------
print("DATA_PATH:", DATA_PATH)
print("DATASET_DIR exists:", DATASET_DIR.exists())
print("Project paths configured successfully.")

DATA_PATH: /content/drive/MyDrive/Master_Research/04_Data/01_Raw_Data/german_credit/german_credit_data.csv
DATASET_DIR exists: True
Project paths configured successfully.


## 7. Load Original Data


In [9]:
# =====================================================
# Load Features Dataset from Kaggle
# =====================================================

df = pd.read_csv(DATA_PATH)
df = df.drop(columns=["Unnamed: 0"]) #Remove index column
print("Features dataset loaded successfully with", df.shape[0], "rows and", df.shape[1], "columns.")



Features dataset loaded successfully with 1000 rows and 9 columns.


In [11]:
# Path
file_path = ROOT_PATH / "04_Data" / "01_Raw_Data" / "german_credit" / "statlog+german+credit+data" / "german.data"

# ---------------------------------------------------------
# Debug check
# ---------------------------------------------------------
print("File exists:", file_path.exists())
print("Full path  :", file_path)

# ---------------------------------------------------------
# Load data (space-separated, no header)
# ---------------------------------------------------------
y = pd.read_csv(file_path, sep=" ", header=None, usecols=[20])

y.head()

File exists: True
Full path  : /content/drive/MyDrive/Master_Research/04_Data/01_Raw_Data/german_credit/statlog+german+credit+data/german.data


,20
0,1
1,2
2,1
3,1
4,2


## Combine Features and Target

In [12]:
# Ensure target column is named consistently
y = y.rename(columns={y.columns[0]: "class"})

# Combine features and target
df["class"] = y["class"].values

print("Combined dataset shape:", df.shape)
df.head()

Combined dataset shape: (1000, 10)


,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,class
0,67,male,2,own,NaN,little,1169,6,radio/TV,1
1,22,female,2,own,little,moderate,5951,48,radio/TV,2
2,49,male,1,own,little,NaN,2096,12,education,1
3,45,male,2,free,little,little,7882,42,furniture/equipment,1
4,53,male,2,free,little,little,4870,24,car,2


## Prepare Target Variable

The target variable `class` represents credit risk.

In the original dataset:

- 1 = Good credit
- 2 = Bad credit

For binary classification modelling, the target is recoded as:

- 0 = Good credit
- 1 = Bad credit

In [13]:
if sorted(df["class"].dropna().unique().tolist()) == [1, 2]:
    df["class"] = df["class"].map({1: 0, 2: 1})

X = df.drop(columns=["class"])
y = df["class"]

print("Target distribution:")
print(y.value_counts(dropna=False))

Target distribution:
class
0    700
1    300
Name: count, dtype: int64


## Identify Feature Types

Features are separated into numeric and categorical groups so that appropriate preprocessing methods can be applied to each type.

In [15]:
numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

Numeric columns: ['Age', 'Job', 'Credit amount', 'Duration']
Categorical columns: ['Sex', 'Housing', 'Saving accounts', 'Checking account', 'Purpose']


## Adjust the Treatment of `Job`

Although `Job` appears as a numeric variable in the dataset, it represents occupational categories rather than continuous numeric measurements.

Therefore, it is treated as a categorical feature.

In [16]:
if "Job" in numeric_cols:
    numeric_cols.remove("Job")
    categorical_cols.append("Job")

print("Updated numeric columns:", numeric_cols)
print("Updated categorical columns:", categorical_cols)

Updated numeric columns: ['Age', 'Credit amount', 'Duration']
Updated categorical columns: ['Sex', 'Housing', 'Saving accounts', 'Checking account', 'Purpose', 'Job']


## Split the Dataset into Training and Testing Sets

The dataset is divided into training and testing subsets.

- 80% training data
- 20% testing data

Stratified sampling is used to preserve the class distribution across both subsets.

In [19]:
# =========================================================
# Train-Test Split (Original Data)
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)

# =========================================================
# Save Original Split
# =========================================================

from pathlib import Path

DATASET_DIR = ROOT_PATH / "07_Results" / "Datasets" / "german_credit"
DATASET_DIR.mkdir(parents=True, exist_ok=True)

# Ensure DataFrame format (important)
X_train_df = X_train.copy()
X_test_df  = X_test.copy()

y_train_df = y_train.to_frame(name="target") if not isinstance(y_train, pd.DataFrame) else y_train
y_test_df  = y_test.to_frame(name="target") if not isinstance(y_test, pd.DataFrame) else y_test

# Save files
X_train_df.to_csv(DATASET_DIR / "X_train_original.csv", index=False)
X_test_df.to_csv(DATASET_DIR / "X_test_original.csv", index=False)

y_train_df.to_csv(DATASET_DIR / "y_train_original.csv", index=False)
y_test_df.to_csv(DATASET_DIR / "y_test_original.csv", index=False)

# ---------------------------------------------------------
# Confirm
# ---------------------------------------------------------
print("Files saved successfully in:", DATASET_DIR)

X_train shape: (800, 9)
X_test shape : (200, 9)
y_train shape: (800,)
y_test shape : (200,)
Files saved successfully in: /content/drive/MyDrive/Master_Research/07_Results/Datasets/german_credit
